In [8]:
import numpy as np
import evaluate
from transformers import BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, pipeline
from datasets import load_dataset

In [9]:
ds = load_dataset("mteb/tweet_sentiment_extraction")
ds

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/240k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26732 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3432 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 26732
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 3432
    })
})

In [10]:
assert len(ds["train"].unique("id")) == len(ds["train"])
assert len(ds["test"].unique("id")) == len(ds["test"])

In [11]:
ds = ds.filter(lambda x: x["text"] is not None and x["label"] is not None)
ds

Filter:   0%|          | 0/26732 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3432 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 26732
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 3432
    })
})

In [12]:
ds_final = ds["train"].train_test_split(test_size = 0.2, seed = 42)
ds_final["validation"] = ds_final.pop("test")
ds_final["test"] = ds["test"]
ds_final

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 21385
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5347
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 3432
    })
})

In [13]:
checkpoint_id = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint_id)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
tokenized_datasets = ds_final.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/21385 [00:00<?, ? examples/s]

Map:   0%|          | 0/5347 [00:00<?, ? examples/s]

Map:   0%|          | 0/3432 [00:00<?, ? examples/s]

In [14]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 21385
    })
    validation: Dataset({
        features: ['id', 'text', 'labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5347
    })
    test: Dataset({
        features: ['id', 'text', 'labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3432
    })
})

In [15]:
print(tokenized_datasets["train"][0].keys())

dict_keys(['id', 'text', 'labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'])


In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(['text', 'label_text', 'id'])

In [35]:
tokenized_datasets.save_to_disk("tokenized_datasets")

Saving the dataset (0/1 shards):   0%|          | 0/21385 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5347 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3432 [00:00<?, ? examples/s]